# Build the analytical database

Load the processed title metadata and experience scores into DuckDB so the final dataset can be queried with SQL.

In [1]:
from pathlib import Path

import pandas as pd
import duckdb


project_root = Path.cwd().parent

metadata_file = (
    project_root
    / "data"
    / "processed"
    / "tmdb_title_metadata.parquet"
)

experience_scores_file = (
    project_root
    / "data"
    / "processed"
    / "experience_scores"
    / "experience_scores.parquet"
)

database_dir = (
    project_root
    / "data"
    / "warehouse"
)

database_dir.mkdir(
    parents=True,
    exist_ok=True
)

database_file = (
    database_dir
    / "movie_experience.duckdb"
)

print("Metadata:", metadata_file)
print("Database:", database_file)
print("Experience scores:", experience_scores_file)

Metadata: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\processed\tmdb_title_metadata.parquet
Database: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\warehouse\movie_experience.duckdb
Experience scores: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\processed\experience_scores\experience_scores.parquet


## Inspect the title metadata

Check the processed TMDB dataset before loading it into the database.

In [2]:
metadata = pd.read_parquet(metadata_file)

print(f"Titles: {len(metadata):,}")
print()
print(metadata.columns.tolist())

metadata.head()

Titles: 7,721

['tmdb_id', 'media_type', 'title', 'original_title', 'overview', 'genres', 'original_language', 'release_date', 'runtime', 'status', 'popularity', 'vote_average', 'vote_count', 'production_countries', 'number_of_seasons', 'number_of_episodes', 'metadata_status']


,tmdb_id,media_type,title,original_title,overview,genres,original_language,release_date,runtime,status,popularity,vote_average,vote_count,production_countries,number_of_seasons,number_of_episodes,metadata_status
0,299534,movie,Avengers: Endgame,Avengers: Endgame,After the devastating events of Avengers: Infi...,"[Adventure, Science Fiction, Action]",en,2019-04-24,181.0,Released,68.0618,8.200,28462,[US],NaN,NaN,success
1,278,movie,The Shawshank Redemption,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"[Drama, Crime]",en,1994-09-23,142.0,Released,75.3973,8.728,31151,[US],NaN,NaN,success
2,645484,movie,Dil Bechara,दिल बेचारा,"Kizie and Manny, two ordinary people brought t...","[Drama, Romance]",hi,2020-07-24,101.0,Released,1.6974,6.776,125,[IN],NaN,NaN,success
3,299537,movie,Captain Marvel,Captain Marvel,The story follows Carol Danvers as she becomes...,"[Action, Adventure, Science Fiction]",en,2019-03-06,124.0,Released,21.6647,6.772,17060,[US],NaN,NaN,success
4,155,movie,The Dark Knight,The Dark Knight,Batman raises the stakes in his war on crime. ...,"[Action, Crime, Thriller]",en,2008-07-16,152.0,Released,57.3845,8.534,36516,"[GB, US]",NaN,NaN,success


## Load the title metadata

Create the title dimension from the processed TMDB metadata. TMDB IDs are stored as integers and the release date is converted to a date type for SQL queries.

In [3]:
# Connect to the DuckDB database
con = duckdb.connect(str(database_file))

# Build the title table with a key that is unique across movies and TV shows
con.execute(
    """
    CREATE OR REPLACE TABLE titles AS
    SELECT
        media_type || ':' || CAST(CAST(tmdb_id AS BIGINT) AS VARCHAR) AS title_key,
        CAST(tmdb_id AS BIGINT) AS tmdb_id,
        media_type,
        title,
        original_title,
        overview,
        genres,
        original_language,
        TRY_CAST(release_date AS DATE) AS release_date,
        TRY_CAST(runtime AS INTEGER) AS runtime,
        status,
        popularity,
        vote_average,
        vote_count,
        production_countries,
        TRY_CAST(number_of_seasons AS INTEGER) AS number_of_seasons,
        TRY_CAST(number_of_episodes AS INTEGER) AS number_of_episodes,
        metadata_status
    FROM read_parquet(?)
    """,
    [str(metadata_file)]
)

print("Titles table created.")

Titles table created.


## Validate the title table

Check that every processed TMDB title was loaded and that each combined movie/TV title key is unique.

In [4]:
con.execute(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT title_key) AS unique_title_keys,
        COUNT(*) - COUNT(DISTINCT title_key) AS duplicate_title_keys,
        SUM(CASE WHEN title_key IS NULL THEN 1 ELSE 0 END) AS missing_title_keys
    FROM titles
    """
).df()

,total_rows,unique_title_keys,duplicate_title_keys,missing_title_keys
0,7721,7721,0,0.0


## Load the experience scores

Load the title-experience profiles created in notebook 05 into DuckDB. The scores are stored in long format, with one row for each title and experience combination.

In [5]:
# Load the final experience scores directly from parquet
con.execute(
    """
    CREATE OR REPLACE TABLE experience_scores AS
    SELECT
        title_key,
        CAST(tmdb_id AS BIGINT) AS tmdb_id,
        media_type,
        experience,
        category,
        CAST(score AS DOUBLE) AS score,
        CAST(raw_similarity AS DOUBLE) AS raw_similarity,
        CAST(reviews_used AS BIGINT) AS reviews_used
    FROM read_parquet(?)
    """,
    [str(experience_scores_file)]
)

print("Experience scores table created.")

Experience scores table created.


## Validate the experience scores

Check that all 7,721 titles have all 55 experience attributes and that no duplicate title-experience combinations were loaded.

In [6]:
con.execute(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT title_key) AS titles,
        COUNT(DISTINCT experience) AS experiences,
        COUNT(*) - COUNT(DISTINCT (title_key, experience)) AS duplicate_pairs,
        SUM(CASE WHEN score IS NULL THEN 1 ELSE 0 END) AS missing_scores
    FROM experience_scores
    """
).df()

,total_rows,titles,experiences,duplicate_pairs,missing_scores
0,424655,7721,55,0,0.0


## Validate the warehouse relationships

Confirm that every experience profile matches a title, that the identifiers agree across both tables, and that every title has all 55 experience scores.

In [7]:
# Check for experience scores that do not match a title
orphan_scores = con.execute(
    """
    SELECT COUNT(*) AS orphan_score_rows
    FROM experience_scores e
    LEFT JOIN titles t
        ON e.title_key = t.title_key
    WHERE t.title_key IS NULL
    """
).fetchone()[0]

# Check that IDs and media types agree across both tables
identifier_mismatches = con.execute(
    """
    SELECT COUNT(*) AS identifier_mismatches
    FROM experience_scores e
    JOIN titles t
        ON e.title_key = t.title_key
    WHERE
        e.tmdb_id <> t.tmdb_id
        OR e.media_type <> t.media_type
    """
).fetchone()[0]

# Check how many experience rows each title has
experience_coverage = con.execute(
    """
    SELECT
        MIN(experience_count) AS min_experiences,
        MAX(experience_count) AS max_experiences
    FROM (
        SELECT
            title_key,
            COUNT(*) AS experience_count
        FROM experience_scores
        GROUP BY title_key
    )
    """
).df()

# Count reviews once per title instead of once per experience
reviews_represented = con.execute(
    """
    SELECT SUM(reviews_used)
    FROM (
        SELECT DISTINCT
            title_key,
            reviews_used
        FROM experience_scores
    )
    """
).fetchone()[0]

print(f"Orphan score rows:       {orphan_scores:,}")
print(f"Identifier mismatches:   {identifier_mismatches:,}")
print(f"Reviews represented:     {reviews_represented:,}")
print()
experience_coverage

Orphan score rows:       0
Identifier mismatches:   0
Reviews represented:     2,723,221



,min_experiences,max_experiences
0,55,55


## Query the experience warehouse

Use the warehouse to retrieve titles based on audience-derived experience scores rather than traditional genre labels. A minimum review count is included in recommendation queries so titles with very little review evidence do not dominate the results.

In [8]:
# Find titles that rank highest for a single experience
comforting_titles = con.execute(
    """
    SELECT
        t.title,
        t.media_type,
        t.release_date,
        e.score,
        e.raw_similarity,
        e.reviews_used
    FROM experience_scores e
    JOIN titles t
        ON e.title_key = t.title_key
    WHERE
        e.experience = 'Comforting'
        AND e.reviews_used >= 100
    ORDER BY e.score DESC
    LIMIT 20
    """
).df()

comforting_titles

,title,media_type,release_date,score,raw_similarity,reviews_used
0,Mentalhood,tv,2020-03-11,100.00,0.551133,637
1,Sweet Sunshine,movie,2020-03-20,99.99,0.547706,104
2,Dosed,movie,2019-12-14,99.97,0.539130,101
3,A Dog's Purpose,movie,2017-01-19,99.96,0.538244,418
4,Butterfly,tv,2018-10-14,99.95,0.537936,119
5,Fisherman's Friends,movie,2019-03-15,99.94,0.535796,108
6,Queer Eye,tv,2018-02-07,99.92,0.534714,112
7,Old Fashioned,movie,2014-10-18,99.91,0.530406,125
8,Snapshots,movie,2018-07-27,99.90,0.530363,115
9,The Lift Boy,movie,2019-01-18,99.88,0.527687,101


### Combine multiple experiences

A recommendation can combine several experience attributes. The scores are averaged here to find titles that perform strongly across the requested experience profile.

In [9]:
# Find titles that combine a comforting, warm, and romantic experience
blended_recommendations = con.execute(
    """
    SELECT
        t.title,
        t.media_type,
        t.release_date,
        ROUND(AVG(e.score), 2) AS experience_match_score,
        MIN(e.reviews_used) AS reviews_used
    FROM experience_scores e
    JOIN titles t
        ON e.title_key = t.title_key
    WHERE
        e.experience IN (
            'Comforting',
            'Warm / tender',
            'Romantic'
        )
        AND e.reviews_used >= 100
    GROUP BY
        t.title_key,
        t.title,
        t.media_type,
        t.release_date
    HAVING COUNT(DISTINCT e.experience) = 3
    ORDER BY experience_match_score DESC
    LIMIT 20
    """
).df()

blended_recommendations

,title,media_type,release_date,experience_match_score,reviews_used
0,Old Fashioned,movie,2014-10-18,99.95,125
1,Sweet Sunshine,movie,2020-03-20,99.93,104
2,Beautiful Thing,movie,1996-06-21,99.91,169
3,Mentalhood,tv,2020-03-11,99.90,637
4,Snapshots,movie,2018-07-27,99.86,115
5,96,movie,2018-10-04,99.85,405
6,Five Feet Apart,movie,2019-03-14,99.70,321
7,"Spring, Summer, Fall, Winter... and Spring",movie,2003-09-19,99.65,197
8,Modern Love,tv,2019-10-18,99.62,220
9,Fisherman's Friends,movie,2019-03-15,99.61,108


### Inspect a title's experience profile

The same warehouse can retrieve the strongest experience attributes for an individual title.

In [10]:
# Show the strongest experience signals for The Dark Knight
dark_knight_profile = con.execute(
    """
    SELECT
        e.experience,
        e.category,
        e.score,
        e.raw_similarity,
        e.reviews_used
    FROM experience_scores e
    WHERE e.title_key = 'movie:155'
    ORDER BY e.score DESC
    LIMIT 10
    """
).df()

dark_knight_profile

,experience,category,score,raw_similarity,reviews_used
0,Unforgettable,After-effect,87.76,0.556148,6815
1,Full of life,Overall experience,78.95,0.514096,6815
2,Epic / grand,Story / world experience,73.40,0.520530,6815
3,Life-affirming,Overall experience,69.95,0.501892,6815
4,Strong chemistry,Connection,62.23,0.543995,6815
5,Immersive,Overall experience,61.05,0.555413,6815
6,Fulfilled,After-effect,51.70,0.510933,6815
7,Human / humanistic,Connection,49.36,0.508101,6815
8,Existential,Thinking / engagement,49.23,0.476952,6815
9,Intense,Overall experience,48.85,0.515592,6815


## Finalize the warehouse

Confirm the completed database tables, then close the DuckDB connection so the warehouse is saved and ready for downstream use.

In [11]:
# Show the completed warehouse tables
con.execute(
    """
    SHOW TABLES
    """
).df()

,name
0,experience_scores
1,titles


In [12]:
# Save all changes and close the database connection
con.close()

print(f"Warehouse saved: {database_file}")

Warehouse saved: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\warehouse\movie_experience.duckdb


## Result

The analytical warehouse contains 7,721 movie and TV titles and 424,655 title-experience scores covering 55 viewing-experience attributes.

The `titles` and `experience_scores` tables are joined through a combined `title_key`, which prevents collisions between movie and TV records that share the same numeric TMDB ID.

The warehouse supports both single-experience ranking and multi-experience recommendation queries. This makes it possible to retrieve titles based on audience-derived signals such as comforting, romantic, immersive, unsettling, or thought-provoking rather than relying only on traditional genre labels.

The experience scores are relative semantic signals rather than ground-truth classifications, so some recommendations may contain noisy matches. Review counts are retained so downstream queries can apply minimum evidence thresholds when needed.